# Build the ordered catalog (45M tracks -> `track_id`-sorted Parquet)

One-time, memory-safe pass that reorders the Embeat 45M catalog by `track_id` so it can be
looked up efficiently.

**Why the special handling (this machine)**
- ~2 GB free RAM and **no swap**, so anything that fills RAM hard-crashes it.
- `/home` has only ~13 GB free, and `/tmp` is a RAM disk (tmpfs).
- So we cap DuckDB memory and push both the sort **spill** and the **output** onto `/mnt/C` (37 GB free).

**What ordering does / does not speed up**
- Great for single `track_id` lookups (row-group pruning).
- For a scattered `IN (~3000 ids)` bulk lookup the main win is columnar projection; ordering is a
  foundation for a later indexed build. See `docs/ARCHITECTURE.md`.

## 1. Configuration

In [1]:
import os, time, duckdb

# Source parquet (read-only, lives on /home)
SOURCE_GLOB = '/home/kala/Documents/rewind/data/metadata/train-*.parquet'

# Scratch + output go on /mnt/C (37 GB free).
# NEVER use /home (~13 GB free) or /tmp (tmpfs = RAM, would crash).
BUILD_DIR   = '/mnt/C/rewind_build'
TEMP_DIR    = os.path.join(BUILD_DIR, 'duckdb_tmp')
OUTPUT_PATH = os.path.join(BUILD_DIR, 'catalog_sorted.parquet')
os.makedirs(TEMP_DIR, exist_ok=True)

# Guardrails: no swap and ~2 GB free RAM, so keep DuckDB small and force disk spill.
MEMORY_LIMIT = '2GB'
THREADS = 2

print('duckdb', duckdb.__version__)
print('source:', SOURCE_GLOB)
print('temp  :', TEMP_DIR)
print('output:', OUTPUT_PATH)

duckdb 1.5.5
source: /home/kala/Documents/rewind/data/metadata/train-*.parquet
temp  : /mnt/C/rewind_build/duckdb_tmp
output: /mnt/C/rewind_build/catalog_sorted.parquet


## 2. Open a memory-capped connection

`temp_directory` is forced onto `/mnt/C` so sorted runs spill to disk, not RAM.

In [2]:
con = duckdb.connect()  # coordinator only; big data streams via disk, not Python
con.execute(f'SET memory_limit={MEMORY_LIMIT!r}')
con.execute(f'SET threads={THREADS}')
con.execute(f'SET temp_directory={TEMP_DIR!r}')
con.execute('SET preserve_insertion_order=false')

for k in ('memory_limit', 'threads', 'temp_directory', 'preserve_insertion_order'):
    v = con.execute(f'SELECT current_setting({k!r})').fetchone()[0]
    print(f'{k:>24} = {v}')

            memory_limit = 1.8 GiB
                 threads = 2
          temp_directory = /mnt/C/rewind_build/duckdb_tmp
preserve_insertion_order = False


## 3. Sanity-check the source (footer metadata only, no rows loaded)

In [3]:
# count(*) reads parquet footer stats only -> no row scan
n_src = con.execute(f'SELECT count(*) FROM read_parquet({SOURCE_GLOB!r})').fetchone()[0]
print(f'source rows: {n_src:,}')

cols = con.execute(f'DESCRIBE SELECT * FROM read_parquet({SOURCE_GLOB!r}) LIMIT 0').fetchall()
print(f'columns: {len(cols)}')

source rows: 45,059,660
columns: 27


## 4. Build the ordered catalog

Streaming external sort: DuckDB reads columnarly, spills sorted runs to `/mnt/C`, and writes a
single `track_id`-ordered Parquet. Nothing is pulled into Python. Expect a few minutes and up to
~9 GB temp plus ~9 GB output on `/mnt/C`.

In [4]:
t0 = time.time()
con.execute(f'''
    COPY (
        SELECT * FROM read_parquet({SOURCE_GLOB!r})
        ORDER BY track_id
    ) TO {OUTPUT_PATH!r}
    (FORMAT PARQUET)
''')
elapsed = time.time() - t0
size_gb = os.path.getsize(OUTPUT_PATH) / 1e9
print(f'sorted + written in {elapsed:,.0f}s  ->  {size_gb:.2f} GB')

sorted + written in 689s  ->  7.53 GB


## 5. Verify (footer stats + tiny reads only)

In [5]:
n_out = con.execute(f'SELECT count(*) FROM read_parquet({OUTPUT_PATH!r})').fetchone()[0]
print(f'output rows: {n_out:,}   source rows: {n_src:,}   match: {n_out == n_src}')

head = con.execute(f'SELECT track_id FROM read_parquet({OUTPUT_PATH!r}) LIMIT 5').fetchall()
print('first 5 track_ids:', [r[0] for r in head])

# Row-group track_id ranges come from the footer only (no row scan)
rg = con.execute(f'''
    SELECT count(*) AS groups, min(stats_min) AS gmin, max(stats_max) AS gmax
    FROM parquet_metadata({OUTPUT_PATH!r})
    WHERE path_in_schema = 'track_id'
''').fetchone()
print(f'row groups: {rg[0]}   track_id range: {rg[1]} -> {rg[2]}')

output rows: 45,059,660   source rows: 45,059,660   match: True
first 5 track_ids: ['01LsWc9dsiW47Jia9KIUoc', '01LsYiARqkAM2yhYkxYNHP', '01Lsa0VKnI2D1IA3psvAqP', '01LsaLSnQF18klyMJIg0GV', '01LsbBscfut7pSoYGnk90h']
row groups: 367   track_id range: 00001tr3U9wKPU5TaAd8v7 -> 7zzzybBJuOEkJDaT7hq0oZ


In [6]:
# Demonstrate a pruned point lookup against the ordered file
sample_id = con.execute(f'SELECT track_id FROM read_parquet({OUTPUT_PATH!r}) LIMIT 1').fetchone()[0]
t0 = time.time()
row = con.execute(f'''
    SELECT track_id, track_name, artist_name, energy, valence, tempo
    FROM read_parquet({OUTPUT_PATH!r})
    WHERE track_id = ?
''', [sample_id]).fetchone()
print(f'point lookup took {(time.time()-t0)*1000:.0f} ms')
print(row)

point lookup took 187 ms
('00001tr3U9wKPU5TaAd8v7', 'Lakambini - Remix', 'Tropical Depression', 0.46700000762939453, 0.640999972820282, 90)


## Done

The ordered catalog is at `/mnt/C/rewind_build/catalog_sorted.parquet`.

**Next steps**
- Point the backend at this path (read-only) for enrichment lookups.
- Optional later: build an indexed DuckDB table (`PRIMARY KEY (track_id)`) for fast scattered bulk
  lookups — best done on the server or a machine with more RAM/disk.
- The original `data/metadata/train-*.parquet` can stay as the cold rebuild source.